# 02 — Building the AI code reviewer

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~40 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/10_code_review_security/02_build.ipynb)

**Prerequisites**:
- **01_architecture.ipynb** — pipeline design, detection prompts, scoring
- OpenAI API key set as `OPENAI_API_KEY` environment variable

**What you will learn**:
- How to build a corpus of 15 Python code samples (10 vulnerable + 5 clean)
- End-to-end AI code-review agent: parse → detect → explain → fix
- AST-based taint tracking with data-flow visualization
- Logic-bug detection for algorithmic errors
- Fix generation with before/after diffs
- Full pull-request review with risk assessment

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "openai>=1.0.0" "pandas>=2.0.0" "numpy>=1.24.0"

import os
import ast
import json
import textwrap
import time
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Set, Any
from collections import defaultdict

import numpy as np
import pandas as pd
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL: str = "gpt-4o-mini"

def chat(system: str, user: str, temperature: float = 0.2) -> str:
    """Send a chat completion request and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    return response.choices[0].message.content.strip()

print(f"Model: {MODEL}")
print("Setup complete ✓")

## Section 1 — Vulnerable code samples

We create a ground-truth corpus of **15 Python files**: 10 with known
vulnerabilities (spanning OWASP Top 10 categories) and 5 that are clean but use
patterns that *look* suspicious to naive detectors. This corpus is both our test
set and the benchmark for evaluating detection accuracy.

In [ ]:
@dataclass
class CodeFile:
    """A code file with ground-truth vulnerability labels."""
    name: str
    code: str
    is_vulnerable: bool
    vuln_class: str                      # e.g., "sql_injection" or "clean"
    severity: str                        # critical | high | medium | low | none
    description: str
    cwe_id: str = ""                     # CWE identifier

# ── 10 VULNERABLE FILES ──
vulnerable_files: List[CodeFile] = [
    CodeFile("auth_login.py", textwrap.dedent('''
        def authenticate(username: str, password: str) -> dict:
            """Authenticate user against the database."""
            query = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
            result = db.execute(query)
            if result:
                return {"authenticated": True, "user": result[0]}
            return {"authenticated": False}
    ''').strip(), True, "sql_injection", "critical",
    "SQL injection in login — attacker can bypass authentication", "CWE-89"),

    CodeFile("file_download.py", textwrap.dedent('''
        import os
        def download(filename: str) -> bytes:
            """Serve a file from the uploads directory."""
            path = os.path.join("/var/uploads", filename)
            with open(path, "rb") as f:
                return f.read()
    ''').strip(), True, "path_traversal", "critical",
    "Path traversal — filename like ../../etc/passwd reads arbitrary files", "CWE-22"),

    CodeFile("config.py", textwrap.dedent('''
        DATABASE_URL = "postgresql://admin:s3cretP@ss@db.internal:5432/prod"
        API_KEY = "sk-proj-abcdef123456789"
        JWT_SECRET = "super-secret-jwt-key-do-not-share"

        def get_db_connection():
            return psycopg2.connect(DATABASE_URL)
    ''').strip(), True, "hardcoded_secrets", "high",
    "Hardcoded database password, API key, and JWT secret", "CWE-798"),

    CodeFile("user_profile.py", textwrap.dedent('''
        def render_profile(user: dict) -> str:
            """Render user profile as HTML."""
            name = user.get("name", "Anonymous")
            bio = user.get("bio", "")
            return f"<div class='profile'><h1>{name}</h1><p>{bio}</p></div>"
    ''').strip(), True, "xss", "high",
    "Reflected XSS — user name and bio rendered without escaping", "CWE-79"),

    CodeFile("data_loader.py", textwrap.dedent('''
        import pickle
        def load_model(data: bytes) -> object:
            """Load a serialized ML model."""
            return pickle.loads(data)
    ''').strip(), True, "insecure_deserialization", "critical",
    "Insecure deserialization — pickle.loads on untrusted data", "CWE-502"),

    CodeFile("webhook.py", textwrap.dedent('''
        import requests
        def fetch_webhook(url: str) -> dict:
            """Fetch data from a user-specified webhook URL."""
            response = requests.get(url, timeout=10)
            return response.json()
    ''').strip(), True, "ssrf", "high",
    "SSRF — user controls URL, can reach internal services", "CWE-918"),

    CodeFile("admin_panel.py", textwrap.dedent('''
        def run_diagnostic(command: str) -> str:
            """Run a diagnostic command on the server."""
            import os
            return os.popen(command).read()
    ''').strip(), True, "command_injection", "critical",
    "Command injection — user input passed directly to os.popen", "CWE-78"),

    CodeFile("template_engine.py", (
    "def render_template(template_str: str, context: dict) -> str:\n"
    "    # Render a user-provided template string using eval\n"
    "    return eval(\"f'\" + template_str + \"'\", {}, context)"
    ), True, "code_injection", "critical",
    "Code injection via eval on user-provided template string", "CWE-94"),

    CodeFile("session_handler.py", textwrap.dedent('''
        import hashlib
        def create_session_token(user_id: str) -> str:
            """Create a session token for the user."""
            return hashlib.md5(user_id.encode()).hexdigest()
    ''').strip(), True, "weak_crypto", "medium",
    "Weak cryptography — MD5 for session tokens is predictable", "CWE-328"),

    CodeFile("log_handler.py", textwrap.dedent('''
        def log_user_action(user_id: str, action: str, details: str) -> None:
            """Log user action with full request details."""
            import logging
            logging.info(f"User {user_id}: {action} - {details}")
            with open("/var/log/app.log", "a") as f:
                f.write(f"{user_id}|{action}|{details}\n")
    ''').strip(), True, "log_injection", "medium",
    "Log injection — unsanitized user input written to log file", "CWE-117"),
]

# ── 5 CLEAN FILES (potential false-positive triggers) ──
clean_files: List[CodeFile] = [
    CodeFile("safe_query.py", textwrap.dedent('''
        def get_user(user_id: int) -> dict:
            """Safely query user by ID with parameterized query."""
            return db.execute("SELECT * FROM users WHERE id = ?", (user_id,))
    ''').strip(), False, "clean", "none",
    "Parameterized query — safe from SQL injection"),

    CodeFile("safe_file_access.py", textwrap.dedent('''
        import os
        ALLOWED_EXTENSIONS = {".pdf", ".png", ".jpg"}
        def safe_download(filename: str) -> bytes:
            """Download with extension whitelist and basename stripping."""
            base = os.path.basename(filename)
            ext = os.path.splitext(base)[1].lower()
            if ext not in ALLOWED_EXTENSIONS:
                raise ValueError(f"Disallowed extension: {ext}")
            path = os.path.join("/var/uploads", base)
            with open(path, "rb") as f:
                return f.read()
    ''').strip(), False, "clean", "none",
    "Path traversal mitigated — basename + extension whitelist"),

    CodeFile("safe_html.py", textwrap.dedent('''
        from markupsafe import escape
        def render_greeting(name: str) -> str:
            """Render greeting with proper HTML escaping."""
            safe_name = escape(name)
            return f"<h1>Hello, {safe_name}!</h1>"
    ''').strip(), False, "clean", "none",
    "XSS mitigated — markupsafe.escape applied"),

    CodeFile("safe_config.py", textwrap.dedent('''
        import os
        DATABASE_URL = os.getenv("DATABASE_URL")
        API_KEY = os.getenv("API_KEY")
        def get_db():
            return psycopg2.connect(DATABASE_URL)
    ''').strip(), False, "clean", "none",
    "Secrets from environment variables — no hardcoded values"),

    CodeFile("safe_serialization.py", textwrap.dedent('''
        import json
        def load_data(raw: str) -> dict:
            """Load data using safe JSON deserialization."""
            return json.loads(raw)
    ''').strip(), False, "clean", "none",
    "JSON deserialization — safe alternative to pickle"),
]

all_files: List[CodeFile] = vulnerable_files + clean_files

print("=" * 70)
print("  CODE CORPUS — 15 FILES")
print("=" * 70)
for f in all_files:
    icon = "🔴" if f.is_vulnerable else "🟢"
    sev = f.severity if f.is_vulnerable else "—"
    print(f"  {icon} {f.name:<25} {f.vuln_class:<25} {sev:<10} {f.cwe_id}")
print(f"\n  Vulnerable: {sum(1 for f in all_files if f.is_vulnerable)}")
print(f"  Clean:      {sum(1 for f in all_files if not f.is_vulnerable)}")

## Section 2 — Code analyzer agent

The core agent takes a code string, analyzes it for security vulnerabilities, classifies
severity, generates an explanation, and suggests a fix — all in one LLM call. We
process all 15 files and collect results for evaluation.

In [ ]:
ANALYZER_SYSTEM: str = textwrap.dedent("""\
    You are an expert application-security code reviewer for Python.
    Analyze the provided code and return a JSON object with:

    {
      "has_vulnerabilities": true/false,
      "findings": [
        {
          "vuln_class": "category (e.g., sql_injection, xss, path_traversal)",
          "severity": "critical | high | medium | low",
          "line": <line_number>,
          "description": "clear explanation of the vulnerability",
          "exploit_scenario": "how an attacker would exploit this",
          "suggested_fix": "corrected code snippet",
          "confidence": "high | medium | low"
        }
      ],
      "summary": "one-sentence overall assessment"
    }

    RULES:
    - Only report genuine vulnerabilities. Parameterized queries, ORM calls,
      proper escaping, and env-var configs are SAFE — do NOT flag them.
    - If the code is clean, set has_vulnerabilities to false and findings to [].
    - Be specific about line numbers and exploitation scenarios.
""")

def analyze_code(file: CodeFile) -> Dict[str, Any]:
    """Run the AI code analyzer on a single file."""
    prompt = f"Filename: {file.name}\n\n```python\n{file.code}\n```"
    raw = chat(ANALYZER_SYSTEM, prompt)
    # Strip markdown code fences if present
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = "\n".join(cleaned.split("\n")[1:])
    if cleaned.endswith("```"):
        cleaned = cleaned[:cleaned.rfind("```")]
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"has_vulnerabilities": None, "findings": [], "summary": raw[:200], "parse_error": True}

# ── Analyze all 15 files ──
print("=" * 70)
print("  AI CODE ANALYZER — PROCESSING 15 FILES")
print("=" * 70)
results: List[Dict[str, Any]] = []
for f in all_files:
    t0 = time.time()
    result = analyze_code(f)
    elapsed = time.time() - t0
    result["filename"] = f.name
    result["ground_truth_vulnerable"] = f.is_vulnerable
    result["ground_truth_class"] = f.vuln_class
    results.append(result)

    detected = result.get("has_vulnerabilities", False)
    icon = "🔴" if detected else "🟢"
    n_findings = len(result.get("findings", []))
    print(f"  {icon} {f.name:<25} detected={str(detected):<6} "
          f"findings={n_findings}  {elapsed:.1f}s")
    if n_findings > 0:
        for finding in result["findings"][:2]:
            print(f"       ↳ {finding.get('vuln_class', '?')}: "
                  f"{finding.get('description', 'N/A')[:60]}")

print(f"\n  Total files analyzed: {len(results)}")
print(f"  Files with findings:  {sum(1 for r in results if r.get('has_vulnerabilities'))}")

## Section 3 — Data flow tracing

We build an enhanced AST-based taint tracker that follows user input through
variable assignments and function calls to identify whether tainted data reaches
a dangerous sink. This deterministic analysis complements the LLM's reasoning.

In [ ]:
class TaintTracker:
    """Enhanced taint tracker using Python AST analysis."""

    SOURCES: Dict[str, str] = {
        "request.args.get": "HTTP query parameter",
        "request.form": "HTTP form data",
        "request.json": "HTTP JSON body",
        "input": "stdin user input",
        "sys.argv": "command-line argument",
    }

    SINKS: Dict[str, str] = {
        "db.execute": "SQL execution",
        "cursor.execute": "SQL execution",
        "os.system": "OS command execution",
        "os.popen": "OS command execution",
        "subprocess.call": "subprocess execution",
        "eval": "code evaluation",
        "exec": "code execution",
        "open": "file system access",
        "pickle.loads": "deserialization",
        "requests.get": "HTTP request (SSRF)",
    }

    def __init__(self) -> None:
        self.tainted_vars: Dict[str, str] = {}   # var_name → source
        self.flows: List[Dict[str, Any]] = []

    def trace(self, code_text: str) -> Dict[str, Any]:
        """Trace tainted data flows through code."""
        self.tainted_vars.clear()
        self.flows.clear()

        lines = code_text.strip().split("\n")
        for i, line in enumerate(lines, 1):
            stripped = line.strip()
            if not stripped or stripped.startswith("#") or stripped.startswith("def "):
                continue

            # Detect sources
            for src, desc in self.SOURCES.items():
                if src in stripped and "=" in stripped:
                    var = stripped.split("=")[0].strip()
                    self.tainted_vars[var] = desc
                    self.flows.append({"line": i, "event": "SOURCE",
                                       "var": var, "detail": f"{var} ← {desc}"})

            # Propagate taint
            if "=" in stripped and not any(stripped.startswith(k) for k in ("if ", "return ", "for ")):
                lhs = stripped.split("=")[0].strip()
                rhs = "=".join(stripped.split("=")[1:])
                for tvar, tsrc in list(self.tainted_vars.items()):
                    if tvar in rhs and lhs != tvar:
                        self.tainted_vars[lhs] = f"derived from {tvar}"
                        self.flows.append({"line": i, "event": "PROPAGATE",
                                           "var": lhs, "detail": f"{lhs} ← f({tvar})"})
                        break

            # Detect sinks
            for sink, desc in self.SINKS.items():
                if sink in stripped:
                    for tvar in self.tainted_vars:
                        if tvar in stripped:
                            self.flows.append({"line": i, "event": "SINK_HIT",
                                               "var": tvar, "detail": f"⚠ {desc} with tainted '{tvar}'"})

        sink_hits = [f for f in self.flows if f["event"] == "SINK_HIT"]
        return {
            "tainted_vars": dict(self.tainted_vars),
            "flows": self.flows,
            "sink_hits": sink_hits,
            "is_vulnerable": len(sink_hits) > 0,
        }

# ── Test on vulnerable samples ──
tracker = TaintTracker()
test_targets: List[CodeFile] = [f for f in vulnerable_files if f.vuln_class in
                                 ("sql_injection", "path_traversal", "ssrf", "command_injection")]

print("=" * 70)
print("  TAINT ANALYSIS — DATA FLOW TRACING")
print("=" * 70)
for f in test_targets:
    result = tracker.trace(f.code)
    print(f"\n  📄 {f.name} ({f.vuln_class})")
    for flow in result["flows"]:
        icon = {"SOURCE": "🟡", "PROPAGATE": "→ ", "SINK_HIT": "🔴"}[flow["event"]]
        print(f"     L{flow['line']:>2} {icon} [{flow['event']:<10}] {flow['detail']}")
    status = "🔴 VULNERABLE" if result["is_vulnerable"] else "🟢 SAFE"
    print(f"     Result: {status}")

# ── Test on clean samples ──
print(f"\n  ── Clean file results ──")
for f in clean_files[:3]:
    result = tracker.trace(f.code)
    status = "🔴 VULNERABLE" if result["is_vulnerable"] else "🟢 SAFE"
    print(f"  {status} {f.name}")

## Section 4 — Logic bug detection

Beyond security vulnerabilities, we detect **algorithmic and logic errors**:
off-by-one in loops, missing null checks, incorrect exception handling, and
unreachable code. These bugs cause wrong results rather than security breaches.

In [ ]:
LOGIC_BUG_SYSTEM: str = textwrap.dedent("""\
    You are an expert code reviewer focused on LOGIC BUGS (not security).
    Analyze the code for correctness errors. Return JSON:
    {
      "has_bugs": true/false,
      "bugs": [
        {
          "bug_type": "off_by_one | null_check | exception_handling | resource_leak | logic_error",
          "severity": "critical | high | medium | low",
          "line": <line_number>,
          "description": "what's wrong",
          "expected_behavior": "what should happen",
          "actual_behavior": "what actually happens",
          "suggested_fix": "corrected code"
        }
      ]
    }
""")

buggy_algorithms: List[Tuple[str, str, str]] = [
    ("binary_search.py", textwrap.dedent('''
        def binary_search(arr: list, target: int) -> int:
            """Return index of target, or -1 if not found."""
            lo, hi = 0, len(arr)
            while lo < hi:
                mid = (lo + hi) // 2
                if arr[mid] == target:
                    return mid
                elif arr[mid] < target:
                    lo = mid
                else:
                    hi = mid
            return -1
    ''').strip(), "Infinite loop when target not found — lo=mid should be lo=mid+1"),

    ("average_calc.py", textwrap.dedent('''
        def calculate_average(numbers: list) -> float:
            """Calculate the average of a list of numbers."""
            total = 0
            for i in range(1, len(numbers)):
                total += numbers[i]
            return total / len(numbers)
    ''').strip(), "Off-by-one: skips first element, divides by full length"),

    ("cache_lookup.py", textwrap.dedent('''
        def get_cached_user(cache: dict, user_id: str) -> str:
            """Get user email from cache. Returns 'unknown' if not cached."""
            user = cache.get(user_id)
            return user["email"]
    ''').strip(), "Missing null check: cache.get returns None if key absent"),

    ("file_processor.py", textwrap.dedent('''
        def process_files(paths: list) -> list:
            """Read and process multiple files."""
            results = []
            for path in paths:
                try:
                    f = open(path)
                    data = f.read()
                    results.append(transform(data))
                except IOError:
                    results.append(None)
            return results
    ''').strip(), "Resource leak: file handle never closed, especially on exception"),

    ("pagination.py", textwrap.dedent('''
        def paginate(items: list, page: int, per_page: int = 10) -> list:
            """Return items for the given page (1-indexed)."""
            start = page * per_page
            end = start + per_page
            return items[start:end]
    ''').strip(), "Off-by-one: page 1 returns items 10-19 instead of 0-9"),
]

print("=" * 70)
print("  LOGIC BUG DETECTION — 5 ALGORITHMS")
print("=" * 70)
logic_results: List[Dict[str, Any]] = []
for name, code_text, ground_truth in buggy_algorithms:
    prompt = f"Filename: {name}\n\n```python\n{code_text}\n```"
    raw = chat(LOGIC_BUG_SYSTEM, prompt)
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = "\n".join(cleaned.split("\n")[1:])
    if cleaned.endswith("```"):
        cleaned = cleaned[:cleaned.rfind("```")]
    try:
        result = json.loads(cleaned)
    except json.JSONDecodeError:
        result = {"has_bugs": None, "bugs": [], "parse_error": True}

    result["filename"] = name
    result["ground_truth"] = ground_truth
    logic_results.append(result)

    has_bugs = result.get("has_bugs", False)
    icon = "🐛" if has_bugs else "✅"
    print(f"\n  {icon} {name}")
    print(f"     Ground truth: {ground_truth[:70]}")
    for bug in result.get("bugs", [])[:2]:
        print(f"     Found: [{bug.get('bug_type', '?')}] {bug.get('description', 'N/A')[:70]}")

detected = sum(1 for r in logic_results if r.get("has_bugs"))
print(f"\n  Detection rate: {detected}/{len(logic_results)} buggy files flagged")

## Section 5 — Fix generator

For every finding — security vulnerability or logic bug — we generate a
**minimal fix** that addresses the issue while preserving existing behavior.
The fix includes before/after code and an explanation.

In [ ]:
FIX_SYSTEM: str = textwrap.dedent("""\
    You are an expert Python developer generating minimal code fixes.
    Given the original code and the identified issue, produce a fix.

    Return JSON:
    {
      "fixed_code": "the complete corrected function/code",
      "changes": ["list of specific changes made"],
      "preserves_behavior": true/false,
      "explanation": "why this fix works"
    }

    RULES:
    - Make the MINIMUM change needed to fix the issue
    - Preserve all existing behavior for valid inputs
    - Follow the original code's style and naming
    - Add a brief comment at the fix point
""")

# ── Generate fixes for first 5 vulnerable files ──
print("=" * 70)
print("  FIX GENERATION — BEFORE / AFTER")
print("=" * 70)
fix_results: List[Dict[str, Any]] = []
for f in vulnerable_files[:5]:
    finding_desc = f.description
    prompt = (f"Original code ({f.name}):\n```python\n{f.code}\n```\n\n"
              f"Issue: {finding_desc}\nVulnerability class: {f.vuln_class}")
    raw = chat(FIX_SYSTEM, prompt)
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = "\n".join(cleaned.split("\n")[1:])
    if cleaned.endswith("```"):
        cleaned = cleaned[:cleaned.rfind("```")]
    try:
        fix_data = json.loads(cleaned)
    except json.JSONDecodeError:
        fix_data = {"fixed_code": "Parse error", "changes": [], "explanation": raw[:200]}

    fix_data["filename"] = f.name
    fix_data["vuln_class"] = f.vuln_class
    fix_results.append(fix_data)

    print(f"\n  ── {f.name} ({f.vuln_class}) ──")
    print(f"  ❌ BEFORE:")
    for line in f.code.split("\n")[:4]:
        print(f"     {line}")
    print(f"  ✅ AFTER:")
    fixed = fix_data.get("fixed_code", "N/A")
    for line in fixed.split("\n")[:4]:
        print(f"     {line}")
    print(f"  Changes: {fix_data.get('changes', ['N/A'])[:2]}")

print(f"\n  Fixes generated: {len(fix_results)}")

## Section 6 — End-to-end pull request review

We simulate a realistic code review scenario: a developer submits a "pull
request" with three changed files. The AI reviewer produces a comprehensive
report including change summary, security findings, logic issues, suggested
fixes, and an overall risk assessment.

In [ ]:
PR_REVIEW_SYSTEM: str = textwrap.dedent("""\
    You are a senior code reviewer conducting a pull request review.
    Analyze ALL changed files together and produce a comprehensive review.

    Return JSON:
    {
      "summary": "one-paragraph summary of the PR changes",
      "security_findings": [
        {"file": "...", "severity": "...", "description": "...", "fix": "..."}
      ],
      "logic_issues": [
        {"file": "...", "severity": "...", "description": "...", "fix": "..."}
      ],
      "style_suggestions": [
        {"file": "...", "suggestion": "..."}
      ],
      "risk_assessment": {
        "overall_risk": "critical | high | medium | low",
        "approval_recommendation": "approve | request_changes | block",
        "reasoning": "..."
      }
    }
""")

# ── Simulated PR: 3 changed files ──
pr_files: Dict[str, str] = {
    "api/endpoints.py": textwrap.dedent('''
        from flask import request, jsonify

        def search_products():
            """Search products by name."""
            query = request.args.get("q", "")
            sql = f"SELECT * FROM products WHERE name LIKE '%{query}%'"
            results = db.execute(sql)
            return jsonify([dict(r) for r in results])

        def get_product(product_id: int):
            """Get a single product by ID."""
            result = db.execute("SELECT * FROM products WHERE id = ?", (product_id,))
            return jsonify(dict(result)) if result else ("Not found", 404)
    ''').strip(),

    "utils/file_handler.py": textwrap.dedent('''
        import os

        def save_upload(filename: str, data: bytes) -> str:
            """Save an uploaded file and return the path."""
            upload_dir = "/var/uploads"
            filepath = os.path.join(upload_dir, filename)
            with open(filepath, "wb") as f:
                f.write(data)
            return filepath

        def read_upload(filename: str) -> bytes:
            filepath = "/var/uploads/" + filename
            return open(filepath, "rb").read()
    ''').strip(),

    "models/user.py": textwrap.dedent('''
        import hashlib, os

        class User:
            def __init__(self, username: str, email: str):
                self.username = username
                self.email = email
                self.password_hash = ""

            def set_password(self, password: str) -> None:
                """Hash and store the password."""
                salt = os.urandom(16).hex()
                self.password_hash = hashlib.sha256(
                    (salt + password).encode()
                ).hexdigest()
                self.salt = salt

            def check_password(self, password: str) -> bool:
                candidate = hashlib.sha256(
                    (self.salt + password).encode()
                ).hexdigest()
                return candidate == self.password_hash
    ''').strip(),
}

# ── Build PR context ──
pr_context = "Pull Request: 'Add product search, file uploads, and user model'\n\n"
for fname, code_text in pr_files.items():
    pr_context += f"--- {fname} ---\n```python\n{code_text}\n```\n\n"

print("=" * 70)
print("  PULL REQUEST REVIEW — 3 FILES")
print("=" * 70)
print(f"  Files: {', '.join(pr_files.keys())}")
print(f"  Total lines: {sum(len(c.split(chr(10))) for c in pr_files.values())}")
print()

t0 = time.time()
raw_review = chat(PR_REVIEW_SYSTEM, pr_context)
elapsed = time.time() - t0

# Parse review
cleaned = raw_review.strip()
if cleaned.startswith("```"):
    cleaned = "\n".join(cleaned.split("\n")[1:])
if cleaned.endswith("```"):
    cleaned = cleaned[:cleaned.rfind("```")]
try:
    review = json.loads(cleaned)
except json.JSONDecodeError:
    review = {"summary": raw_review[:300], "security_findings": [],
              "logic_issues": [], "risk_assessment": {"overall_risk": "unknown"}}

print(f"  📋 SUMMARY: {review.get('summary', 'N/A')[:120]}")
print()

sec = review.get("security_findings", [])
print(f"  🔒 SECURITY FINDINGS ({len(sec)}):")
for finding in sec:
    sev = finding.get("severity", "?")
    sev_icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}.get(sev, "⚪")
    print(f"     {sev_icon} [{sev}] {finding.get('file', '?')}: {finding.get('description', 'N/A')[:70]}")

logic = review.get("logic_issues", [])
print(f"\n  🐛 LOGIC ISSUES ({len(logic)}):")
for issue in logic:
    print(f"     • {issue.get('file', '?')}: {issue.get('description', 'N/A')[:70]}")

risk = review.get("risk_assessment", {})
risk_level = risk.get("overall_risk", "unknown")
approval = risk.get("approval_recommendation", "unknown")
risk_icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}.get(risk_level, "⚪")
print(f"\n  {risk_icon} RISK: {risk_level}  |  RECOMMENDATION: {approval}")
print(f"     {risk.get('reasoning', 'N/A')[:100]}")
print(f"\n  Review completed in {elapsed:.1f}s")

In [ ]:
# ── Build summary ──
vuln_detected = sum(1 for r in results if r.get("has_vulnerabilities"))
clean_correct = sum(1 for r in results
                    if not r["ground_truth_vulnerable"]
                    and not r.get("has_vulnerabilities"))
logic_detected = sum(1 for r in logic_results if r.get("has_bugs"))

print("=" * 60)
print("  BUILD SUMMARY")
print("=" * 60)
print(f"  Corpus:              {len(all_files)} files (10 vuln + 5 clean)")
print(f"  Vulns detected:      {vuln_detected}/10")
print(f"  Clean correct:       {clean_correct}/5")
print(f"  Logic bugs found:    {logic_detected}/5")
print(f"  Fixes generated:     {len(fix_results)}")
print(f"  PR review:           complete")

## Exercises

1. **Add SAST comparison**: Run the regex-based detector from notebook 00 on all
   15 files. Compare its precision/recall to the AI analyzer. Which files does
   each approach handle better?

2. **Multi-language support**: Adapt the code analyzer prompt to handle
   JavaScript code. Create 3 vulnerable JS samples (prototype pollution, eval
   injection, path traversal) and test the analyzer.

3. **Incremental review**: Modify the PR review to work on diffs instead of full
   files — given only the changed lines plus 5 lines of context. Does this reduce
   tokens while maintaining detection accuracy?

## Takeaways

- A corpus of **10 vulnerable + 5 clean** files provides a balanced benchmark for detection
- The AI analyzer achieves high detection rates with **lower false positives** than regex
- **Taint tracking** catches multi-hop data flows that even LLMs may miss on first pass
- Logic-bug detection requires understanding **programmer intent** — LLMs reason about this well
- Fix generation must be **minimal and behavior-preserving** to earn developer trust
- End-to-end PR review combines all stages into an **actionable risk assessment**

## What's next

In **03_evaluate.ipynb** we rigorously measure the reviewer's performance:
detection precision/recall, false-positive analysis, fix correctness, severity
calibration, and cost/efficiency calculations.

## References

1. OWASP Top 10 (2021) — https://owasp.org/www-project-top-ten/
2. CWE/SANS Top 25 — https://cwe.mitre.org/top25/
3. "AI-Powered Code Review" — GitHub Universe 2023
4. Semgrep Rules Registry — https://semgrep.dev/r
5. "Automated Vulnerability Detection with LLMs" — Thapa et al., 2023